# Phase 1: Baseline Classifier with Spatially-Disjoint Split (Colab GPU)

This Google Colab notebook runs the Phase 1 training and evaluation pipeline on GPU acceleration.

### Pipeline Steps:
1. Check GPU availability (`!nvidia-smi`)
2. Download EuroSAT dataset & install dependencies
3. Generate spatially-disjoint perceptual hash split vs random split (`data/split.py`)
4. Train ResNet50 & EfficientNet-B0 classifiers (`model/train.py`)
5. Evaluate accuracy, per-class F1 score, and plot confusion matrix heatmaps (`model/evaluate.py`)

In [ ]:
# 1. Verify GPU availability
!nvidia-smi
import torch
print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Install requirements
!pip install -q timm imagehash scikit-learn kagglehub matplotlib tqdm pandas

In [ ]:
# 3. Download EuroSAT Dataset
from data.download import download_eurosat
dataset_path = download_eurosat(use_kaggle=True)
print("EuroSAT dataset ready at:", dataset_path)

In [ ]:
# 4. Generate Random & Spatial Cluster Splits
from data.split import generate_and_save_splits
splits_data = generate_and_save_splits()
print("Random Split counts:", splits_data["random_split"]["train_count"], splits_data["random_split"]["test_count"])
print("Spatial Split counts:", splits_data["spatial_split"]["train_count"], splits_data["spatial_split"]["test_count"])

In [ ]:
# 5. Train ResNet-50 Classifier on Spatial Split (GPU Accelerated)
from model.train import run_training
results_spatial = run_training(model_name="resnet50", split_type="spatial", epochs=5, batch_size=64, lr=1e-3)

In [ ]:
# 6. Train ResNet-50 Classifier on Random Split for Generalization Gap Comparison
results_random = run_training(model_name="resnet50", split_type="random", epochs=5, batch_size=64, lr=1e-3)

In [ ]:
# 7. Evaluate Performance & Plot Confusion Matrices
from model.evaluate import compare_split_metrics
summary = compare_split_metrics(model_name="resnet50")